In [ ]:

import sqlite3
import pandas as pd
from IPython.display import display, HTML
import ipywidgets as widgets

db_path = "sjur_recortes.db"

def carregar_dataframe(query):
    with sqlite3.connect(db_path) as conn:
        return pd.read_sql_query(query, conn)


In [ ]:

df_emails = carregar_dataframe("SELECT id, message_id, data_processamento FROM emails_processados ORDER BY id DESC")
display(HTML("<h3>📧 Emails Processados</h3>"))
display(df_emails)


In [ ]:

seletor_email = widgets.Dropdown(
    options=[(f"{row['id']} | {row['message_id'][:40]}...", row['id']) for _, row in df_emails.iterrows()],
    description='Email ID:',
    layout=widgets.Layout(width='100%')
)
display(seletor_email)


In [ ]:

def mostrar_detalhes(email_id):
    html = f"<hr><h4>📌 Detalhes para Email ID: {email_id}</h4>"
    display(HTML(html))

    # Tabela: publicacoes
    df_pub = carregar_dataframe(f"SELECT texto, tipo FROM publicacoes WHERE message_id = (SELECT message_id FROM emails_processados WHERE id = {email_id})")
    if not df_pub.empty:
        display(HTML("<b>🗞️ Publicações:</b>"))
        display(df_pub)
    else:
        display(HTML("<i>⚠️ Nenhuma publicação encontrada.</i>"))

    # Tabela: partes
    df_partes = carregar_dataframe(f"SELECT parte, papel FROM partes WHERE message_id = (SELECT message_id FROM emails_processados WHERE id = {email_id})")
    if not df_partes.empty:
        display(HTML("<b>👥 Partes:</b>"))
        display(df_partes)
    else:
        display(HTML("<i>⚠️ Nenhuma parte encontrada.</i>"))

    # Tabela: metadados
    df_meta = carregar_dataframe(f"SELECT chave, valor FROM metadados WHERE message_id = (SELECT message_id FROM emails_processados WHERE id = {email_id})")
    if not df_meta.empty:
        display(HTML("<b>🗂️ Metadados:</b>"))
        display(df_meta)
    else:
        display(HTML("<i>⚠️ Nenhum metadado encontrado.</i>"))


In [ ]:

botao = widgets.Button(description="🔍 Ver Detalhes", button_style='primary')

def on_click(b):
    mostrar_detalhes(seletor_email.value)

botao.on_click(on_click)
display(botao)
